In [1]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = (_here if os.path.isdir(os.path.join(_here, "pipeline"))
         else os.path.abspath(os.path.join(_here, os.pardir)))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print("project root:", _root)

project root: /home/temari/god please no diploma/restore_punct


In [2]:
EXAMPLES = [
    (
        "Мама мыла раму а папа читал газету",
        "Мама мыла раму, а папа читал газету.",
    ),
    (
        "Однако мы решили не идти в кино потому что пошел дождь",
        "Однако мы решили не идти в кино, потому что пошёл дождь.",
    ),
    (
        "Он сказал Привет как дела",
        'Он сказал: "Привет, как дела?"',
    ),
    (
        "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
        'Я говорю ей: "Что думаешь?" А она мне: "Да ничего, отстань уже".',
    ),
    (
        "После многих страданий А А Пушкин всё-таки написал свои стишки",
        "После многих страданий А. А. Пушкин всё-таки написал свои стишки.",
    ),
    (
        "Почему вы сидите сказал Вася Да мы просто устали ответила Маша",
        '- Почему вы сидите? - сказал Вася. - Да мы просто устали, - ответила Маша.',
    ),
    (
        "Где был этот ваш будильник когда я спал",
        "Где был этот ваш будильник, когда я спал?",
    ),
    (
        "Ну как же так неужели никто не пришёл",
        "Ну как же так, неужели никто не пришёл?",
    ),
    (
        "Да ладно тебе всё будет хорошо сказала мать",
        '- Да ладно тебе, всё будет хорошо, - сказала мать.',
    ),
    (
        "во-первых нужно было подготовиться во-вторых собрать документы",
        "Во-первых, нужно было подготовиться; во-вторых, собрать документы.",
    ),
    (
        "Москва столица России а Санкт-Петербург культурная столица",
        "Москва - столица России, а Санкт-Петербург - культурная столица.",
    ),
    (
        "Она весело бежала по полю вглядываясь в буйство красок: ромашки, колокольчики, ландыши, васильки, и одуванчики - рассыпались, по земле, словно цветные пятна на палитре.",
        "Она весело бежала по полю, вглядываясь в буйство красок: ромашки, колокольчики, ландыши, васильки и одуванчики рассыпались по земле, словно цветные пятна на палитре.",
    ),
    (
        "Я не знаю ответил он тихо может быть завтра",
        '- Я не знаю, - ответил он тихо, - Может быть, завтра.',
    ),
    (
        "Солнце село за горизонт и наступила тишина удивительная и глубокая",
        "Солнце село за горизонт, и наступила тишина - удивительная и глубокая.",
    ),
]

print(f"Total examples: {len(EXAMPLES)}")

Total examples: 14


In [3]:
import json
import os
from pipeline.env import RESULTS_DIR

with open(os.path.join(RESULTS_DIR, "models_db.json")) as f:
    models_db = json.load(f)

MODEL_LIST = []
for name, entry in models_db.items():
    arch = entry.get("config", {}).get("architecture", "bert+crf")
    MODEL_LIST.append((name, arch))

print(f"Models to evaluate ({len(MODEL_LIST)}):")
for name, arch in MODEL_LIST:
    print(f"  {name:40s}  [{arch}]")

Models to evaluate (16):
  01_baseline_ce                            [bert]
  01_baseline_crf                           [bert+crf]
  01_baseline_focal                         [bert]
  02_baseline_crf_transitions               [bert+crf]
  02_baseline_focal_mask_o                  [bert]
  02_baseline_ce_mask_o                     [bert]
  01_baseline_crf_focal                     [bert+crf]
  03_crf_transitions_synthetic              [bert+crf]
  03_crf_transitions_mined                  [bert+crf]
  03_crf_transitions_mined_5000             [bert+crf]
  04a_gera_lorugec_mixed                    [bert+crf]
  04c_finetune_no_replay                    [bert+crf]
  04b_finetune_replay                       [bert+crf]
  04b_finetune_replay_v2                    [bert+crf]
  03_crf_transitions_mined_v2               [bert+crf]
  04b_finetune_replay_v3                    [bert+crf]


In [4]:
import gc
import torch
from pipeline.config import RunConfig
from pipeline.inference import load_for_inference, restore_punctuation

results = {}

for model_name, arch in MODEL_LIST:
    print(f"\n{'='*60}")
    print(f"Loading: {model_name}  ({arch})")
    print(f"{'='*60}")

    cfg = RunConfig(name=model_name, architecture=arch)

    try:
        model, tokenizer = load_for_inference(cfg)
    except FileNotFoundError as e:
        print(f"  SKIP (not found): {e}")
        results[model_name] = ["[model not found]"] * len(EXAMPLES)
        continue

    restored = []
    for stripped, _true in EXAMPLES:
        out = restore_punctuation(model, tokenizer, stripped)
        restored.append(out)
        print(f"  IN:  {stripped[:60]}...")
        print(f"  OUT: {out[:60]}...")

    results[model_name] = restored

    del model
    del tokenizer
    del cfg
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"  GPU freed. VRAM free: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")

print("\nAll models done.")


Loading: 01_baseline_ce  (bert)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  IN:  Мама мыла раму а папа читал газету...
  OUT: Мама мыла раму, а папа читал газету....
  IN:  Однако мы решили не идти в кино потому что пошел дождь...
  OUT: Однако мы решили не идти в кино, потому что пошел дождь....
  IN:  Он сказал Привет как дела...
  OUT: Он сказал: " Привет, как дела"....
  IN:  Я говорю ей Что думаешь А она мне Да ничего отстань уже...
  OUT: Я говорю ей: " Что думаешь? А она мне! Да ничего отстань уже...
  IN:  После многих страданий А А Пушкин всё-таки написал свои стиш...
  OUT: После многих страданий А. А. Пушкин всё-таки написал свои ст...
  IN:  Почему вы сидите сказал Вася Да мы просто устали ответила Ма...
  OUT: Почему вы сидите? - сказал Вася. - Да, мы просто устали", - ...
  IN:  Где был этот ваш будильник когда я спал...
  OUT: Где был этот ваш будильник, когда я спал?...
  IN:  Ну как же так неужели никто не пришёл...
  OUT: Ну как же так неужели никто не пришёл?...
  IN:  Да ладно тебе всё будет хорошо сказала мать...
  OUT: Да ладно, тебе вс

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:305: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


...

  IN:  Мама мыла раму а папа читал газету...
  OUT: Мама мыла раму, а папа читал газету....
  IN:  Однако мы решили не идти в кино потому что пошел дождь...
  OUT: Однако мы решили не идти в кино, потому что пошел дождь....
  IN:  Он сказал Привет как дела...
  OUT: Он сказал: " Привет, как дела"....
  IN:  Я говорю ей Что думаешь А она мне Да ничего отстань уже...
  OUT: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань...
  IN:  После многих страданий А А Пушкин всё-таки написал свои стиш...
  OUT: После многих страданий А. А. Пушкин всё-таки написал свои ст...
  IN:  Почему вы сидите сказал Вася Да мы просто устали ответила Ма...
  OUT: Почему вы сидите? - сказал Вася. - Да, мы просто устали, - о...
  IN:  Где был этот ваш будильник когда я спал...
  OUT: Где был этот ваш будильник, когда я спал?...
  IN:  Ну как же так неужели никто не пришёл...
  OUT: Ну как же так неужели никто не пришёл?...
  IN:  Да ладно тебе всё будет хорошо сказала мать...
  OUT: Да ладно, тебе вс

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  IN:  Мама мыла раму а папа читал газету...
  OUT: Мама мыла раму, а папа читал газету....
  IN:  Однако мы решили не идти в кино потому что пошел дождь...
  OUT: Однако мы решили не идти в кино, потому что пошел дождь....
  IN:  Он сказал Привет как дела...
  OUT: Он сказал: " Привет, как дела"....
  IN:  Я говорю ей Что думаешь А она мне Да ничего отстань уже...
  OUT: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань...
  IN:  После многих страданий А А Пушкин всё-таки написал свои стиш...
  OUT: После многих страданий А. А. Пушкин всё-таки написал свои ст...
  IN:  Почему вы сидите сказал Вася Да мы просто устали ответила Ма...
  OUT: Почему вы сидите? - сказал Вася. - Да мы просто устали, - от...
  IN:  Где был этот ваш будильник когда я спал...
  OUT: Где был этот ваш будильник, когда я спал?...
  IN:  Ну как же так неужели никто не пришёл...
  OUT: Ну как же так, неужели никто не пришёл?...
  IN:  Да ладно тебе всё будет хорошо сказала мать...
  OUT: Да ладно, тебе в

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  IN:  Мама мыла раму а папа читал газету...
  OUT: Мама мыла раму, а папа читал газету....
  IN:  Однако мы решили не идти в кино потому что пошел дождь...
  OUT: Однако мы решили не идти в кино, потому что пошел дождь....
  IN:  Он сказал Привет как дела...
  OUT: Он сказал: " Привет, как дела"....
  IN:  Я говорю ей Что думаешь А она мне Да ничего отстань уже...
  OUT: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань...
  IN:  После многих страданий А А Пушкин всё-таки написал свои стиш...
  OUT: После многих страданий А. А. Пушкин всё-таки написал свои ст...
  IN:  Почему вы сидите сказал Вася Да мы просто устали ответила Ма...
  OUT: Почему вы сидите? - сказал Вася. - Да мы просто устали, - от...
  IN:  Где был этот ваш будильник когда я спал...
  OUT: Где был этот ваш будильник, когда я спал?...
  IN:  Ну как же так неужели никто не пришёл...
  OUT: Ну как же так, неужели никто не пришёл?...
  IN:  Да ладно тебе всё будет хорошо сказала мать...
  OUT: Да ладно, тебе в

In [5]:
import time
import requests

secrets_path = os.path.join(_root, "yandex_secrets.json")
with open(secrets_path, "r", encoding="utf-8") as f:
    _secrets = json.load(f)
    API_KEY = _secrets["YANDEX_API_KEY"]
    FOLDER_ID = _secrets["YANDEX_FOLDER_ID"]

def prompt_yandex_to_restore(unpunctuated_text):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}",
        "x-folder-id": FOLDER_ID,
    }
    body = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.0,
            "maxTokens": "4000",
        },
        "messages": [
            {
                "role": "system",
                "text": (
                    "Ты — профессиональный редактор русского языка. "
                    "Твоя задача — расставить знаки препинания в тексте, где они полностью удалены. "
                    "Строго соблюдай правила:\n"
                    "1. Ни в коем случае не изменяй, не удаляй и не добавляй новые слова. "
                    "Не исправляй регистр букв. Только вставляй пунктуацию.\n"
                    "2. Не объединяй и не разделяй слова через дефис.\n"
                    "3. Выведи только восстановленный текст, без каких-либо комментариев и кавычек."
                ),
            },
            {"role": "user", "text": "Мама мыла раму а папа читал газету"},
            {"role": "assistant", "text": "Мама мыла раму, а папа читал газету."},
            {"role": "user", "text": "Он сказал что придет завтра а она ответила Хорошо"},
            {"role": "assistant", "text": 'Он сказал, что придет завтра, а она ответила: "Хорошо".'},
            {"role": "user", "text": unpunctuated_text},
        ],
    }
    try:
        response = requests.post(url, headers=headers, json=body)
        if response.status_code == 200:
            return response.json()["result"]["alternatives"][0]["message"]["text"]
        else:
            print(f"  API Error {response.status_code}: {response.text[:200]}")
            return None
    except Exception as e:
        print(f"  Request failed: {e}")
        return None

YANDEX_MODEL_NAME = "YandexGPT"
yandex_restored = []

print(f"{'='*60}")
print(f"Prompting: {YANDEX_MODEL_NAME}")
print(f"{'='*60}")

for stripped, _true in EXAMPLES:
    out = prompt_yandex_to_restore(stripped)
    if out is None:
        out = "[API error]"
    yandex_restored.append(out)
    print(f"  IN:  {stripped[:70]}")
    print(f"  OUT: {out[:70]}")
    time.sleep(0.6)

results[YANDEX_MODEL_NAME] = yandex_restored
MODEL_LIST.append((YANDEX_MODEL_NAME, "yandexgpt"))
print("\nYandexGPT done.")

Prompting: YandexGPT
  IN:  Мама мыла раму а папа читал газету
  OUT: Мама мыла раму, а папа читал газету.
  IN:  Однако мы решили не идти в кино потому что пошел дождь
  OUT: Однако мы решили не идти в кино, потому что пошёл дождь.
  IN:  Он сказал Привет как дела
  OUT: Он сказал: «Привет, как дела?»
  IN:  Я говорю ей Что думаешь А она мне Да ничего отстань уже
  OUT: Я говорю ей: "Что думаешь?" А она мне: "Да ничего, отстань уже."
  IN:  После многих страданий А А Пушкин всё-таки написал свои стишки
  OUT: После многих страданий А. А. Пушкин всё-таки написал свои стишки.
  IN:  Почему вы сидите сказал Вася Да мы просто устали ответила Маша
  OUT: «Почему вы сидите?» — сказал Вася. «Да, мы просто устали», — ответила 
  IN:  Где был этот ваш будильник когда я спал
  OUT: Где был этот ваш будильник, когда я спал?
  IN:  Ну как же так неужели никто не пришёл
  OUT: Ну как же так, неужели никто не пришёл?
  IN:  Да ладно тебе всё будет хорошо сказала мать
  OUT: Да ладно тебе, всё будет

In [6]:
import pandas as pd

rows = []
for i, (stripped, true_punct) in enumerate(EXAMPLES):
    row = {
        "Input (stripped)": stripped,
        "True punctuation": true_punct,
    }
    for model_name, _arch in MODEL_LIST:
        row[model_name] = results[model_name][i]
    rows.append(row)

df = pd.DataFrame(rows)

out_path = os.path.join(RESULTS_DIR, "demo_all_models.xlsx")
df.to_excel(out_path, index=False, engine="openpyxl")
print(f"Saved -> {out_path}")
print(f"Shape: {df.shape}")
df

Saved -> /home/temari/god please no diploma/restore_punct/results/demo_all_models.xlsx
Shape: (14, 19)


,Input (stripped),True punctuation,01_baseline_ce,01_baseline_crf,01_baseline_focal,02_baseline_crf_transitions,02_baseline_focal_mask_o,02_baseline_ce_mask_o,01_baseline_crf_focal,03_crf_transitions_synthetic,03_crf_transitions_mined,03_crf_transitions_mined_5000,04a_gera_lorugec_mixed,04c_finetune_no_replay,04b_finetune_replay,04b_finetune_replay_v2,03_crf_transitions_mined_v2,04b_finetune_replay_v3,YandexGPT
0,Мама мыла раму а папа читал газету,"Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла"" раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету.","Мама мыла раму, а папа читал газету."
1,Однако мы решили не идти в кино потому что пош...,"Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по...","Однако мы решили не идти в кино, потому что по..."
2,Он сказал Привет как дела,"Он сказал: ""Привет, как дела?""","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела.","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: "" Привет, как дела"".","Он сказал: «Привет, как дела?»"
3,Я говорю ей Что думаешь А она мне Да ничего от...,"Я говорю ей: ""Что думаешь?"" А она мне: ""Да нич...","Я говорю ей: "" Что думаешь? А она мне! Да ниче...","Я говорю ей: "" Что думаешь А она мне Да ничего...","Я говорю ей: "" Что? думаешь? А она- мне: "" Да,...","Я говорю ей: "" Что думаешь? А она мне Да ничег...","Я говорю ей: "" Что? думаешь? А она- мне: "" Да,...","Я говорю ей: "" Что думаешь? А она мне! Да ниче...","Я говорю ей: "" Что думаешь? А она мне Да ничег...","Я говорю ей: "" Что думаешь? А она мне Да ничег...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь? А она мне Да ничег...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: "" Что думаешь?"" А она мне: "" Да н...","Я говорю ей: ""Что думаешь?"" А она мне: ""Да нич..."
4,После многих страданий А А Пушкин всё-таки нап...,После многих страданий А. А. Пушкин всё-таки н...,После многих страданий А. А. Пушкин всё-таки н...,После мн

In [7]:
from IPython.display import HTML

def highlight_diff(true_val, pred_val):
    if true_val.strip() == pred_val.strip():
        return f'<span style="color:green">{pred_val}</span>'
    return f'<span style="color:red">{pred_val}</span>'

html_parts = []
for i, (stripped, true_punct) in enumerate(EXAMPLES):
    html_parts.append(f"<h4>Example {i+1}</h4>")
    html_parts.append(f"<b>Input:</b> {stripped}<br>")
    html_parts.append(f"<b>True:</b> {true_punct}<br>")
    for model_name, _ in MODEL_LIST:
        pred = results[model_name][i]
        colored = highlight_diff(true_punct, pred)
        html_parts.append(f"<b>{model_name}:</b> {colored}<br>")
    html_parts.append("<hr>")

HTML("".join(html_parts))